# U-AMP Cellular Attention — Final Tournament
This Colab compares the current **Adaptive+MaxPool** champion against four controlled architectural upgrades at the same `feature_dim=32`.

Candidates:
1. `cellular_adaptive_maxpool5` — current champion/control
2. `cellular_adaptive_maxpool5_rms` — gated RMSNorm
3. `cellular_adaptive_mixedpool5_rms` — attention + MaxPool + MeanPool + RMSNorm
4. `cellular_uamp5` — causal two-level encoder/decoder U-Net + gated skips + SwiGLU
5. `cellular_uamp5_channelgate` — U-AMP + causal prefix channel gating

Selection uses validation NLL only. Held-out tests remain separate. Conventional BatchNorm is intentionally not used because cross-token/batch statistics are a poor fit for autoregressive causality.


In [ ]:
import os, sys, subprocess, tempfile, pathlib
REPO_URL = "https://github.com/vtavakkoli/TinyCeNN-LM.git"
repo = pathlib.Path(tempfile.mkdtemp(prefix="tinycenn-uamp-")) / "TinyCeNN-LM"
subprocess.run(["git","clone","--depth","1","--branch","main",REPO_URL,str(repo)], check=True)
os.chdir(repo)
subprocess.run([sys.executable,"-m","pip","install","-q","-e",".",
                "transformers==4.57.6","datasets>=3,<5","huggingface_hub>=0.34,<2",
                "pandas","matplotlib","pytest>=8"], check=True)
print("Repository:", repo)
print("Commit:", subprocess.check_output(["git","rev-parse","HEAD"], text=True).strip())


In [ ]:
import torch, os
PROFILE = "balanced"
LAYERS = "18"
FEATURE_DIMS = "32"
SEED = 2026
SAVE_TO_DRIVE = False
VARIANTS = ",".join([
    "cellular_adaptive_maxpool5",
    "cellular_adaptive_maxpool5_rms",
    "cellular_adaptive_mixedpool5_rms",
    "cellular_uamp5",
    "cellular_uamp5_channelgate",
])

assert torch.cuda.is_available(), "GPU required. In Colab: Runtime > Change runtime type > T4 GPU."
print("GPU:", torch.cuda.get_device_name(0))
print("Variants:", VARIANTS)

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = "/content/drive/MyDrive/TinyCeNN-UAMP-final-results"
else:
    OUTPUT_ROOT = "/content/TinyCeNN-UAMP-final-results"
os.makedirs(OUTPUT_ROOT, exist_ok=True)


In [ ]:
import subprocess, sys
subprocess.run([
    sys.executable, "-m", "pytest", "-q",
    "tests/test_cellular_attention.py",
    "tests/test_cellular_attention_cli.py",
], check=True)
print("✅ Cellular Attention and launcher tests passed")


In [ ]:
import subprocess, sys, os
cmd = [
    sys.executable, "-u", "scripts/run_cellular_attention_colab.py",
    "--profile", PROFILE,
    "--layers", LAYERS,
    "--feature-dims", FEATURE_DIMS,
    "--variants", VARIANTS,
    "--seed", str(SEED),
    "--output-root", OUTPUT_ROOT,
]
print("Running:", " ".join(cmd))
result = subprocess.run(cmd, cwd=repo)
print("Launcher exit code:", result.returncode)


In [ ]:
import json, pathlib
status_path = pathlib.Path(OUTPUT_ROOT) / "last_run.json"
status = json.loads(status_path.read_text())
print(json.dumps(status, indent=2))
if status.get("status") != "completed":
    print("\n--- failure tail ---\n")
    print(status.get("error_tail", "No error tail was recorded."))
    raise RuntimeError(f"Tournament failed at status={status.get('status')}")
RUN_DIR = pathlib.Path(status["output_dir"])
print("✅ Results:", RUN_DIR)


In [ ]:
import pandas as pd, json, numpy as np
validation = pd.read_csv(RUN_DIR / "validation_summary.csv")
summary = pd.read_csv(RUN_DIR / "cellular_attention_summary.csv")
selection = json.loads((RUN_DIR / "selection.json").read_text())
winner = selection["winners"]["18"]

rank = validation.sort_values("validation_nll")[
    ["candidate","variant","feature_dim","validation_nll","validation_delta_nll",
     "validation_output_nmse","validation_output_cosine","trainable_parameters","training_seconds"]
].reset_index(drop=True)
rank.insert(0, "rank", range(1, len(rank)+1))
display(rank)

print("\n🏆 Validation-selected winner:", winner)
winner_test = summary[summary["candidate"].eq(winner)][
    ["context","test_nll","test_perplexity","transformer_perplexity","delta_nll",
     "delta_nll_ci_low","delta_nll_ci_high","ppl_ratio","score_pair_ratio",
     "prefill_ms","transformer_prefill_ms","peak_extra_bytes","quality"]
].sort_values("context")
display(winner_test)

controls = summary[
    summary["candidate"].eq("transformer_original") |
    summary["candidate"].eq(winner) |
    summary["variant"].eq("cellular_adaptive_maxpool5")
][["candidate","variant","context","test_nll","test_perplexity","delta_nll","ppl_ratio","quality"]]
display(controls.sort_values(["context","delta_nll"]))


In [ ]:
import matplotlib.pyplot as plt
plot = validation.sort_values("validation_delta_nll")
plt.figure(figsize=(10,5))
plt.barh(plot["variant"], plot["validation_delta_nll"])
plt.axvline(0.0, linewidth=1)
plt.xlabel("Validation ΔNLL vs Transformer (lower is better)")
plt.title("U-AMP tournament — validation ranking")
plt.tight_layout()
plt.show()

test = summary[summary["candidate"].eq(winner)].sort_values("context")
plt.figure(figsize=(7,4))
plt.plot(test["context"], test["ppl_ratio"], marker="o", label="winner / Transformer")
plt.axhline(1.0, linewidth=1)
plt.xlabel("Context")
plt.ylabel("Perplexity ratio")
plt.title("Validation-selected winner on held-out test")
plt.tight_layout()
plt.show()


In [ ]:
from pathlib import Path
import shutil
from google.colab import files
zip_base = Path("/content/UAMP-final-tournament-results")
zip_path = Path(shutil.make_archive(str(zip_base), "zip", root_dir=RUN_DIR))
print("ZIP:", zip_path)
files.download(str(zip_path))
